In [ ]:
# A demo of a LangChain, then LangGraph agent

In [2]:
%%time
import time
# From the previous notebook
# --------------- LANGCHAIN + OLLAMA ---------------
from langchain_community.llms import Ollama
from langchain.embeddings import OllamaEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import TextLoader
from langchain.chains import RetrievalQA

loader = TextLoader("climate.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
docs = splitter.split_documents(raw_docs)

embeddings = OllamaEmbeddings(model="mxbai-embed-large")
vs = Chroma.from_documents(docs, embeddings, persist_directory="lc_chroma")
retriever = vs.as_retriever(search_type="mmr")

llm = Ollama(model="llama3")
qa = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)
print("[LangChain]", qa.run("What causes the most CO2 emissions?"))
# --------------- LANGCHAIN + OLLAMA -------------------

# Load the required LangChain components
from langchain_community.llms import Ollama                    # Local LLM interface (e.g., llama3)
from langchain.embeddings import OllamaEmbeddings              # Embedding model via Ollama (e.g., mxbai-embed-large)
from langchain.vectorstores import Chroma                      # Vector store to store and search chunk embeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter  # Splits long text into manageable chunks
from langchain.document_loaders import TextLoader              # Utility to load text files as documents
from langchain.chains import RetrievalQA                       # Retrieval-augmented QA pipeline

# ------------------- Perception: Load and Prepare Knowledge -------------------

# Load the raw document from file
loader = TextLoader("climate.txt")
raw_docs = loader.load()  # List of Document objects

# Split the document into smaller overlapping chunks for better semantic matching
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
docs = splitter.split_documents(raw_docs)

# ------------------- Representation: Embed and Store Chunks -------------------

# Load a local embedding model via Ollama (can be changed to any compatible embedding model)
embeddings = OllamaEmbeddings(model="mxbai-embed-large")

# Create or load a Chroma vector store from the split chunks
vs = Chroma.from_documents(docs, embeddings, persist_directory="lc_chroma")

# Create a retriever using Max Marginal Relevance (MMR) to reduce redundancy
retriever = vs.as_retriever(search_type="mmr", search_kwargs={"k": 4})

# ------------------- Reasoning: Setup the LLM -------------------

# Load the local language model (e.g., LLaMA 3 via Ollama)
llm = Ollama(model="llama3")

# Wrap everything in a RetrievalQA chain: retrieve relevant chunk, pass to LLM, and get answer
qa = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

# ------------------- Action: Ask a Question -------------------

# Run the full agentic pipeline
print("[LangChain]", qa.invoke("What causes the most CO2 emissions?"))




[LangChain] According to the context, the largest contributor to CO2 emissions is the burning of fossil fuels for energy and transportation. This includes coal-fired power plants, gasoline-powered vehicles, and industrial processes like cement production. Deforestation also contributes by reducing the planet's ability to absorb CO2.
[LangChain] {'query': 'What causes the most CO2 emissions?', 'result': "The largest contributor to CO2 emissions is the burning of fossil fuels for energy and transportation. This includes coal-fired power plants, gasoline-powered vehicles, and industrial processes like cement production. Deforestation also contributes by reducing the planet's ability to absorb CO2."}
CPU times: user 1.51 s, sys: 855 ms, total: 2.36 s
Wall time: 7.4 s


This LangChain RAG setup behaves like an agent in a sandbox: it perceives its environment (a fixed document), chooses the most relevant information (via retrieval), and takes a single goal-driven action (answering the question). It’s not a fully autonomous agent — but it captures the core structure of one.

Let's increase the agentic side, by leveraging a proper agent call, and providing to the agent the choice between several tools.

In [9]:
%%time
from langchain.agents import Tool, initialize_agent
from langchain.agents.agent_types import AgentType
from langchain.llms import Ollama
from langchain.utilities import WikipediaAPIWrapper
from langchain.chains import RetrievalQA
from langchain.vectorstores import Chroma
from langchain.embeddings import OllamaEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import TextLoader

# ============================================================
# 1. Load and embed document
# ============================================================

loader = TextLoader("climate.txt")
docs = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=100
).split_documents(loader.load())

vs = Chroma.from_documents(
    docs,
    OllamaEmbeddings(model="mxbai-embed-large"),
    persist_directory="lc_agent_chroma"
)
retriever = vs.as_retriever(search_type="mmr", search_kwargs={"k": 4})

llm = Ollama(model="llama3", temperature=0.0)

rag_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)
wiki_tool = WikipediaAPIWrapper()

# ============================================================
# 2. Define tools
# ============================================================

tools = [
    Tool(
        name="LocalDocQA",
        func=rag_chain.run,
        description="Use this to answer questions based on the local climate document."
    ),
    Tool(
        name="WikipediaSearch",
        func=wiki_tool.run,
        description="Use this for information not found in the local document."
    )
]

# ============================================================
# 3. Strict LLaMA-friendly ReAct system prompt with termination guarantee
# ============================================================

react_system_prompt = """
You are a ReAct agent with access to the following tools:

- LocalDocQA
- WikipediaSearch

UNTIL you know the final answer, follow EXACTLY this format:

Thought: <your reasoning>
Action: <one of LocalDocQA or WikipediaSearch>
Action Input: <the input for the tool>

When you have enough information to answer the user's question:
YOU MUST STOP USING TOOLS AND OUTPUT:

Final Answer: <the final answer>

Hard rules:
- NEVER output "Action: None".
- NEVER invent new tools.
- NEVER repeat previous steps.
- NEVER loop.
- NEVER continue reasoning after Final Answer.
- NEVER output Thought or Action after Final Answer.
- ALWAYS end with a line starting with exactly: Final Answer:
- If you make a formatting mistake, FIX IT and output the correct format.
"""

# ============================================================
# 4. Auto-repair for malformed LLaMA ReAct output
# ============================================================

def repair_react_output(text: str) -> str:
    """Repair common incorrect patterns in LLaMA ReAct output."""
    # Fix wrong Action
    if "Action: None" in text:
        text = text.replace("Action: None", "Action: LocalDocQA")
        if "Action Input:" not in text:
            text += '\nAction Input: "climate information"'

    # Fix missing Action Input
    if "Action:" in text and "Action Input:" not in text and "Final Answer" not in text:
        text += '\nAction Input: ""'

    # If answer is present but not formatted
    if "Final" in text and "Final Answer:" not in text:
        idx = text.lower().find("final")
        answer = text[idx:]
        text = f"Final Answer: {answer}"

    return text


# ============================================================
# 5. Ultra-reliable output parser that forces Final Answer
# ============================================================

from langchain.agents.output_parsers import ReActSingleInputOutputParser

class ReliableReActParser(ReActSingleInputOutputParser):
    def parse(self, text):
        # If Final Answer already present — return directly
        if "Final Answer:" in text:
            fa = text.split("Final Answer:", 1)[1].strip()
            return {"output": fa}

        # Try normal parsing
        try:
            return super().parse(text)
        except Exception:
            # Attempt repair
            repaired = repair_react_output(text)

            # If repair includes Final Answer — return it
            if "Final Answer:" in repaired:
                fa = repaired.split("Final Answer:", 1)[1].strip()
                return {"output": fa}

            # Force final answer as last resort
            return {"output": f"Final Answer: {repaired}"}


# ============================================================
# 6. Initialize reliable agent
# ============================================================

agent_executor = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=False,
    handle_parsing_errors=True,
    system_message=react_system_prompt,
    output_parser=ReliableReActParser(),
    max_iterations=8,            # prevents loops
    max_execution_time=120       # adjust as needed
)

# ============================================================
# 7. Run query
# ============================================================

response = agent_executor.run("What causes the most CO2 emissions?")
print("\n[Agentic LangChain Result]", response)




[Agentic LangChain Result] The most significant cause of CO2 emissions is the burning of fossil fuels for energy and transportation, which includes coal-fired power plants, gasoline-powered vehicles, and industrial processes like cement production. Deforestation also contributes by reducing the planet's ability to absorb CO2.

Note: I've combined the information from both LocalDocQA and WikipediaSearch actions to provide a comprehensive answer.
CPU times: user 2.66 s, sys: 1.2 s, total: 3.86 s
Wall time: 1min


What happens in the background? Let's expose the call that the "agent" program makes, the responses of the LLM, and the loops that occur until the LLM surfaces the expected answer.

In [8]:
%%time
from langchain.agents import Tool, initialize_agent
from langchain.agents.agent_types import AgentType
from langchain.llms import Ollama
from langchain.utilities import WikipediaAPIWrapper
from langchain.chains import RetrievalQA
from langchain.vectorstores import Chroma
from langchain.embeddings import OllamaEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import TextLoader

# ============================================================
# 1. Load and embed document
# ============================================================

loader = TextLoader("climate.txt")
docs = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100).split_documents(loader.load())

vs = Chroma.from_documents(
    docs,
    OllamaEmbeddings(model="mxbai-embed-large"),
    persist_directory="lc_agent_chroma"
)
retriever = vs.as_retriever(search_type="mmr", search_kwargs={"k": 4})

llm = Ollama(model="llama3", temperature=0.0)

rag_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)
wiki_tool = WikipediaAPIWrapper()

# ============================================================
# 2. Tools
# ============================================================

tools = [
    Tool(
        name="LocalDocQA",
        func=rag_chain.run,
        description="Use this tool to answer questions based on the local climate document."
    ),
    Tool(
        name="WikipediaSearch",
        func=wiki_tool.run,
        description="Use this for information not found in the local document."
    )
]

# ============================================================
# 3. STRICT ReAct system prompt (with guaranteed termination)
# ============================================================

react_system_prompt = """
You are a ReAct agent with access to the following tools:

- LocalDocQA
- WikipediaSearch

UNTIL you know the final answer, follow EXACTLY this format:

Thought: <your reasoning>
Action: <one of LocalDocQA or WikipediaSearch>
Action Input: <the input for the tool>

When you have enough information to answer the user's question:
YOU MUST STOP USING TOOLS AND OUTPUT:

Final Answer: <the final answer>

Hard rules:
- NEVER output "Action: None".
- NEVER invent new tools.
- NEVER repeat previous steps.
- NEVER loop.
- NEVER continue reasoning after Final Answer.
- NEVER output Thought or Action after Final Answer.
- ALWAYS end with a line starting exactly with: Final Answer:
- If you make a formatting mistake, FIX IT and output the correct format.
"""

# ============================================================
# 4. Repair malformed LLaMA ReAct output
# ============================================================

def repair_react_output(text: str) -> str:
    """Repair common incorrect patterns in LLaMA ReAct output."""
    if "Action: None" in text:
        text = text.replace("Action: None", "Action: LocalDocQA")
        if "Action Input:" not in text:
            text += '\nAction Input: "climate information"'

    if "Action:" in text and "Action Input:" not in text and "Final Answer" not in text:
        text += '\nAction Input: ""'

    if "Final" in text and "Final Answer:" not in text:
        idx = text.lower().find("final")
        answer = text[idx:]
        text = f"Final Answer: {answer}"

    return text


# ============================================================
# 5. Ultra-safe parser that forces Final Answer
# ============================================================

from langchain.agents.output_parsers import ReActSingleInputOutputParser

class ReliableReActParser(ReActSingleInputOutputParser):
    def parse(self, text):
        # Final Answer already provided → return directly
        if "Final Answer:" in text:
            return {"output": text.split("Final Answer:", 1)[1].strip()}

        try:
            return super().parse(text)
        except Exception:
            repaired = repair_react_output(text)

            if "Final Answer:" in repaired:
                return {"output": repaired.split("Final Answer:", 1)[1].strip()}

            # Last fallback
            return {"output": f"Final Answer: {repaired}"}


# ============================================================
# 6. Create the agent
# ============================================================

agent_executor = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    system_message=react_system_prompt,
    output_parser=ReliableReActParser(),
    max_iterations=8,
    max_execution_time=120
)

# ============================================================
# 7. DEBUGGING: Show Internals
# ============================================================

print("\n=== (1) Base ReAct Prompt Template ===")
print(agent_executor.agent.llm_chain.prompt.template)

# Render the prompt exactly as the LLM sees it
formatted_prompt = agent_executor.agent.llm_chain.prompt.format_prompt(
    input="What causes the most CO2 emissions?",
    tools=tools,
    tool_names=[tool.name for tool in tools],
    agent_scratchpad=""
)

print("\n=== (2) FULL Rendered Prompt Sent to LLaMA ===")
print(formatted_prompt.to_string())

# ============================================================
# 8. Run Query
# ============================================================

print("\n=== (3) Starting Agent Execution ===\n")
response = agent_executor.run("What causes the most CO2 emissions?")

print("\n=== (4) Final Answer ===")
print(response)




=== (1) Base ReAct Prompt Template ===
Answer the following questions as best you can. You have access to the following tools:

LocalDocQA(*args: Any, callbacks: Union[list[langchain_core.callbacks.base.BaseCallbackHandler], langchain_core.callbacks.base.BaseCallbackManager, NoneType] = None, tags: Optional[list[str]] = None, metadata: Optional[dict[str, Any]] = None, **kwargs: Any) -> Any - Use this tool to answer questions based on the local climate document.
WikipediaSearch(query: str) -> str - Use this for information not found in the local document.

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [LocalDocQA, WikipediaSearch]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input ques

## Building a Custom Agent with LangGraph (Step-by-Step)

This section demonstrates how to build a **simple, transparent agent** using **LangGraph**, designed to mimic a LangChain agent — but with far more control over structure, logic, visibility, and execution flow.

LangGraph avoids all hidden loops, scratchpads, or output parsers.  
You explicitly define each reasoning step, making the agent predictable, explainable, and fast.

---

## Step 1: Load and Embed Documents

This step prepares the data and tools that the agent can use.

**Components Used:**
- `TextLoader` to read `climate.txt`
- `RecursiveCharacterTextSplitter` to split the text into semantic chunks
- `OllamaEmbeddings` (`mxbai-embed-large`) to generate vector representations
- `Chroma` to store and retrieve embeddings
- `Ollama` (local `llama3`) as the language model
- `RetrievalQA` to combine retriever + LLM into a local RAG tool
- `WikipediaAPIWrapper` as the fallback tool

**Purpose:**  
This initializes all capabilities the agent may use when answering questions.

---

## Step 2: Define the Agent's State

LangGraph requires a structured state object to specify what data flows through the graph.

**`QAState` includes:**
- `question`: The user’s input
- `rag_answer`: Output from the RetrievalQA tool
- `wiki_answer`: Output from the Wikipedia tool
- `answer`: The final answer returned by the agent  
  *(Marked with `Annotated[..., "output"]` so LangGraph knows it is the final output.)*

**Purpose:**  
Defines the memory slots shared across all nodes.

---

## Step 3: Router Node

This is the agent’s “decision-maker.”

### `tool_selector(state)`:
- Prints the question (verbose mode)
- Determines whether the question is climate-related
- Returns:
  - `{"__next__": "rag"}` for local RAG  
  - `{"__next__": "wiki"}` for Wikipedia

**Purpose:**  
Implements a transparent, controllable routing policy instead of relying on an LLM’s interpretation.

---

## Step 4: Tool Nodes

Each node corresponds to a tool call and prints internal details.

### `rag_node(state)`
- Prints input question  
- Runs the RetrievalQA tool  
- Stores result in `rag_answer`

### `wiki_node(state)`
- Prints input question  
- Runs the Wikipedia tool  
- Stores result in `wiki_answer`

**Purpose:**  
Executes concrete actions depending on the router’s choice.

---

## Step 5: Merger Node

This node consolidates tool outputs and determines what the final answer is.

### `merger(state)`
- Prints both RAG and Wikipedia outputs  
- Chooses:
  - `rag_answer` if available  
  - otherwise `wiki_answer`
- Writes the result into `answer`

**Purpose:**  
Ensures the graph always produces a clean, final answer.

---

## Step 6: Build the Graph

We assemble the agent as a directed graph:

1. Create graph:  
   `graph = StateGraph(QAState)`

2. Add nodes:
   - `"router"`
   - `"rag"`
   - `"wiki"`
   - `"merger"`

3. Set entry point:  
   `graph.set_entry_point("router")`

4. Connect edges:
   - `router → rag`
   - `router → wiki`
   - `rag → merger`
   - `wiki → merger`
   - `merger → END`

5. Compile agent:  
   `agent = graph.compile()`

**Purpose:**  
Defines the exact execution path for the agent, fully transparent and testable.

---

## Step 7: Run the Agent

You invoke the agent with:

```python
question = "What causes the most CO2 emissions?"
result = agent.invoke({"question": question})


In [13]:
%%time
# ---------------- LangGraph Agentic Example ----------------

from typing import TypedDict, Optional, Annotated
from langchain_community.llms import Ollama
from langchain.embeddings import OllamaEmbeddings
from langchain.vectorstores import Chroma
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain.utilities import WikipediaAPIWrapper
from langchain_core.runnables import RunnableLambda
from langgraph.graph import StateGraph, END


# ============================================================
# 1. Load and embed documents
# ============================================================

print("\n=== STEP 1: Loading documents ===")
loader = TextLoader("climate.txt")
docs = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100).split_documents(loader.load())

vs = Chroma.from_documents(
    docs,
    OllamaEmbeddings(model="mxbai-embed-large"),
    persist_directory="graph_chroma"
)
retriever = vs.as_retriever(search_kwargs={"k": 4})

llm = Ollama(model="llama3", temperature=0.0)

rag_tool = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)
wiki_tool = WikipediaAPIWrapper()


# ============================================================
# 2. State
# ============================================================

class QAState(TypedDict):
    question: str
    rag_answer: Optional[str]
    wiki_answer: Optional[str]
    answer: Annotated[Optional[str], "output"]


# ============================================================
# 3. Router Node  (Verbose)
# ============================================================

def tool_selector(state: QAState):
    q = state["question"]
    print("\n=== ROUTER NODE ===")
    print("Question:", q)

    lower = q.lower()
    if ("climate" in lower or "co2" in lower or "emission" in lower):
        print("→ Routing to: RAG")
        return {"__next__": "rag"}
    else:
        print("→ Routing to: WIKIPEDIA")
        return {"__next__": "wiki"}


# ============================================================
# 4. Tool Nodes (Verbose)
# ============================================================

def rag_node(state: QAState):
    print("\n=== RAG NODE ===")
    print("Input question:", state["question"])

    out = rag_tool.run(state["question"])

    print("RAG Output:", out)
    return {"rag_answer": out}


def wiki_node(state: QAState):
    print("\n=== WIKI NODE ===")
    print("Input question:", state["question"])

    out = wiki_tool.run(state["question"])

    print("Wikipedia Output:", out)
    return {"wiki_answer": out}


# ============================================================
# 5. Merger Node (Verbose)
# ============================================================

def merger(state: QAState):
    print("\n=== MERGER NODE ===")
    print("RAG Answer:", state.get("rag_answer"))
    print("Wiki Answer:", state.get("wiki_answer"))

    final = state.get("rag_answer") or state.get("wiki_answer")
    print("→ FINAL ANSWER:", final)

    return {"answer": final}


# ============================================================
# 6. Build the Graph
# ============================================================

graph = StateGraph(QAState)

graph.add_node("router", RunnableLambda(tool_selector))
graph.add_node("rag", RunnableLambda(rag_node))
graph.add_node("wiki", RunnableLambda(wiki_node))
graph.add_node("merger", RunnableLambda(merger))

graph.set_entry_point("router")

graph.add_edge("router", "rag")
graph.add_edge("router", "wiki")
graph.add_edge("rag", "merger")
graph.add_edge("wiki", "merger")
graph.add_edge("merger", END)

agent = graph.compile()


# ============================================================
# 7. Run it
# ============================================================

question = "What causes the most CO2 emissions?"

print("\n\n=== RUNNING LANGGRAPH AGENT ===")
result = agent.invoke({"question": question})

print("\n\n=== FINAL RESULT ===")
print(result["answer"])




=== STEP 1: Loading documents ===


=== RUNNING LANGGRAPH AGENT ===

=== ROUTER NODE ===
Question: What causes the most CO2 emissions?
→ Routing to: RAG

=== RAG NODE ===
Input question: What causes the most CO2 emissions?

=== WIKI NODE ===
Input question: What causes the most CO2 emissions?
Wikipedia Output: Page: Carbon dioxide in the atmosphere of Earth
Summary: In the atmosphere of Earth, carbon dioxide is a trace gas that plays an integral part in the greenhouse effect, carbon cycle, photosynthesis, and oceanic carbon cycle. It is one of three main greenhouse gases in the atmosphere of Earth. The concentration of carbon dioxide (CO2) in the atmosphere reached 427 ppm (0.0427%) on a molar basis in 2024, representing 3341 gigatonnes of CO2. This is an increase of 50% since the start of the Industrial Revolution, up from 280 ppm during the 10,000 years prior to the mid-18th century. The increase is due to human activity.
The current increase in CO2 concentrations is primarily drive